# Phase 6 - Final Export and Dashboard/Backend 

This notebook:

1. creates small dashboard-ready analytical tables;
2. copies the final feature and visualization contracts;
3. creates a model manifest describing every saved model;
4. validates that the saved models and dashboard artifacts can actually be loaded and used.

In [1]:
import os
import json
from datetime import datetime

import joblib
import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

os.makedirs('artifacts/dashboard', exist_ok=True)
os.makedirs('reports/handoff', exist_ok=True)

print("Working folder:", os.getcwd())

Working folder: C:\dev\Moodwave\moodwave-ml


## 1. Load the final processed datasets

In [2]:
if os.path.exists('data/processed/historical_tracks.parquet'):
    historical = pd.read_parquet('data/processed/historical_tracks.parquet')
    genre = pd.read_parquet('data/processed/genre_tracks.parquet')
    country = pd.read_parquet('data/processed/country_chart_observations.parquet')
    track_catalog = pd.read_parquet('data/processed/track_catalog.parquet')
else:
    historical = pd.read_csv('data/processed/historical_tracks.csv.gz')
    genre = pd.read_csv('data/processed/genre_tracks.csv.gz')
    country = pd.read_csv('data/processed/country_chart_observations.csv.gz')
    track_catalog = pd.read_csv('data/processed/track_catalog.csv.gz')

country['snapshot_date'] = pd.to_datetime(country['snapshot_date'], errors='coerce')

print('Historical:', historical.shape)
print('Genre:', genre.shape)
print('Country:', country.shape)
print('Track catalog:', track_catalog.shape)

Historical: (2400, 28)
Genre: (113550, 27)
Country: (2110316, 32)
Track catalog: (115657, 24)


## 2. Load the Phase 5 cluster and similarity artifacts

In [3]:
if os.path.exists('artifacts/dashboard/cluster_points.parquet'):
    cluster_points = pd.read_parquet('artifacts/dashboard/cluster_points.parquet')
    similarity_catalog = pd.read_parquet('artifacts/dashboard/similarity_catalog.parquet')
else:
    cluster_points = pd.read_csv('artifacts/dashboard/cluster_points.csv.gz')
    similarity_catalog = pd.read_csv('artifacts/dashboard/similarity_catalog.csv.gz')

with open('artifacts/dashboard/cluster_profiles.json') as f:
    cluster_profiles = json.load(f)

print('Cluster points:', cluster_points.shape)
print('Similarity catalog:', similarity_catalog.shape)
print('Cluster profiles:', len(cluster_profiles))

Cluster points: (115657, 11)
Similarity catalog: (115657, 7)
Cluster profiles: 5


## 3. Yearly mood trends

This table is designed for the future decade/year timeline. It prevents the backend from recomputing the same group-by operation for every request.

In [4]:
yearly_mood_trends = (
    historical
    .groupby('year')
    .agg(
        track_count=('track_id', 'size'),
        mean_valence=('valence', 'mean'),
        mean_energy=('energy', 'mean'),
        mean_danceability=('danceability', 'mean'),
        mean_popularity=('popularity', 'mean')
    )
)

yearly_mood_pct = pd.crosstab(
    historical['year'],
    historical['mood_label'],
    normalize='index'
) * 100

yearly_mood_pct = yearly_mood_pct.rename(columns={
    'Euphoric': 'pct_euphoric',
    'Peaceful': 'pct_peaceful',
    'Aggressive': 'pct_aggressive',
    'Melancholic': 'pct_melancholic'
})

yearly_mood_trends = yearly_mood_trends.join(yearly_mood_pct).reset_index()

yearly_mood_trends.head()

,year,track_count,mean_valence,mean_energy,mean_danceability,mean_popularity,pct_aggressive,pct_euphoric,pct_melancholic,pct_peaceful
0,2000,100,0.628990,0.749500,0.65875,65.66,18.0,71.0,6.0,5.0
1,2001,100,0.653694,0.720960,0.67281,66.79,14.0,76.0,7.0,3.0
2,2002,100,0.574905,0.729480,0.65076,64.64,36.0,59.0,3.0,2.0
3,2003,100,0.600036,0.705130,0.66311,66.25,22.0,66.0,7.0,5.0
4,2004,100,0.598722,0.712545,0.66997,67.68,25.0,60.0,7.0,8.0


## 4. Genre profiles

All genre labels remain available in the exported table. The dashboard can choose a readable subset without losing the underlying information.

In [5]:
genre_audio_features = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo'
]

genre_profiles = (
    genre
    .groupby('track_genre')
    .agg(
        track_count=('track_id', 'size'),
        unique_tracks=('track_id', 'nunique'),
        mean_popularity=('popularity', 'mean'),
        **{
            'mean_' + feature: (feature, 'mean')
            for feature in genre_audio_features
        }
    )
)

genre_mood_pct = pd.crosstab(
    genre['track_genre'],
    genre['mood_label'],
    normalize='index'
) * 100

genre_mood_pct = genre_mood_pct.rename(columns={
    'Euphoric': 'pct_euphoric',
    'Peaceful': 'pct_peaceful',
    'Aggressive': 'pct_aggressive',
    'Melancholic': 'pct_melancholic'
})

genre_profiles = genre_profiles.join(genre_mood_pct).reset_index()

print('Genres exported:', len(genre_profiles))
genre_profiles.head()

Genres exported: 114


,track_genre,track_count,unique_tracks,mean_popularity,mean_danceability,mean_energy,mean_loudness,mean_speechiness,mean_acousticness,mean_instrumentalness,mean_liveness,mean_valence,mean_tempo,pct_aggressive,pct_euphoric,pct_melancholic,pct_peaceful
0,acoustic,1000,1000,42.483000,0.549593,0.435368,-9.447843,0.043247,0.566816,0.038336,0.153244,0.424023,119.010624,15.500000,20.200000,50.900000,13.400000
1,afrobeat,999,999,24.407407,0.669408,0.702938,-7.789599,0.086622,0.270893,0.252876,0.184724,0.698475,119.242057,13.213213,72.772773,5.305305,8.708709
2,alt-rock,999,999,33.896897,0.534601,0.754027,-6.194009,0.055076,0.122168,0.054151,0.210340,0.518179,124.654403,39.739740,51.151151,6.706707,2.402402
3,alternative,999,999,24.361361,0.559963,0.719848,-6.076191,0.070134,0.147967,0.037501,0.201441,0.495158,122.207769,40.740741,44.344344,10.410410,4.504505
4,ambient,999,999,44.208208,0.367966,0.237257,-18.596266,0.041553,0.776701,0.675993,0.129298,0.167345,111.156413,11.111111,1.801802,85.585586,1.501502


## 5. Country/date mood trends

The grain is one row per `country + snapshot_date`. `GLOBAL` remains a valid World Top chart scope rather than being treated as missing.

In [6]:
country_group_columns = ['country', 'snapshot_date']

country_mood_trends = (
    country
    .groupby(country_group_columns)
    .agg(
        track_count=('track_id', 'size'),
        mean_popularity=('popularity', 'mean'),
        mean_daily_rank=('daily_rank', 'mean'),
        mean_valence=('valence', 'mean'),
        mean_energy=('energy', 'mean'),
        mean_danceability=('danceability', 'mean')
    )
)

country_mood_pct = pd.crosstab(
    [country['country'], country['snapshot_date']],
    country['mood_label'],
    normalize='index'
) * 100

country_mood_pct = country_mood_pct.rename(columns={
    'Euphoric': 'pct_euphoric',
    'Peaceful': 'pct_peaceful',
    'Aggressive': 'pct_aggressive',
    'Melancholic': 'pct_melancholic'
})

country_mood_trends = country_mood_trends.join(country_mood_pct).reset_index()

print('Country/date rows exported:', len(country_mood_trends))
country_mood_trends.head()

Country/date rows exported: 42199


,country,snapshot_date,track_count,mean_popularity,mean_daily_rank,mean_valence,mean_energy,mean_danceability,pct_aggressive,pct_euphoric,pct_melancholic,pct_peaceful
0,AE,2023-10-18,50,86.16,25.5,0.48960,0.629082,0.65458,42.0,42.0,12.0,4.0
1,AE,2023-10-19,50,86.18,25.5,0.48960,0.629082,0.65458,42.0,42.0,12.0,4.0
2,AE,2023-10-20,50,88.20,25.5,0.50740,0.643862,0.65206,42.0,46.0,8.0,4.0
3,AE,2023-10-21,50,90.22,25.5,0.49130,0.636522,0.65664,44.0,44.0,8.0,4.0
4,AE,2023-10-22,50,87.10,25.5,0.49294,0.648162,0.65954,46.0,44.0,8.0,2.0


## 6. Export dashboard-ready tables

Parquet is the preferred format because it is smaller, faster and preserves data types. CSV.GZ mirrors are also written so the tables remain easy to inspect manually.

In [7]:
dashboard_tables = {
    'yearly_mood_trends': yearly_mood_trends,
    'genre_profiles': genre_profiles,
    'country_mood_trends': country_mood_trends,
    'cluster_points': cluster_points,
    'similarity_catalog': similarity_catalog,
    'track_catalog': track_catalog
}

parquet_created = True

for name, table in dashboard_tables.items():
    table.to_csv(
        'artifacts/dashboard/' + name + '.csv.gz',
        index=False,
        compression='gzip'
    )

    try:
        table.to_parquet(
            'artifacts/dashboard/' + name + '.parquet',
            index=False
        )
    except ImportError:
        parquet_created = False

if parquet_created:
    print('Parquet and CSV.GZ dashboard tables saved.')
else:
    print('CSV.GZ tables saved. Install PyArrow to also create the preferred Parquet copies.')

Parquet and CSV.GZ dashboard tables saved.


## 7. Export the shared feature and visualization contracts

In [8]:
with open('config/feature_schema.json') as f:
    feature_config = json.load(f)

with open('config/mood_definition.json') as f:
    mood_definition = json.load(f)

# Strict field rules that a later backend can use to validate user input.
dashboard_feature_schema = {
    'fields': {
        'danceability': {'type': 'float', 'min': 0.0, 'max': 1.0},
        'energy': {'type': 'float', 'min': 0.0, 'max': 1.0},
        'key': {'type': 'integer', 'min': 0, 'max': 11},
        'loudness': {'type': 'float'},
        'mode': {'type': 'integer', 'allowed': [0, 1]},
        'speechiness': {'type': 'float', 'min': 0.0, 'max': 1.0},
        'acousticness': {'type': 'float', 'min': 0.0, 'max': 1.0},
        'instrumentalness': {'type': 'float', 'min': 0.0, 'max': 1.0},
        'liveness': {'type': 'float', 'min': 0.0, 'max': 1.0},
        'valence': {'type': 'float', 'min': 0.0, 'max': 1.0},
        'tempo': {'type': 'float', 'min_exclusive': 0.0},
        'duration_ms': {'type': 'integer', 'min_exclusive': 0},
        'time_signature': {'type': 'integer'}
    },
    'audio_features': feature_config['audio_features'],
    'mood_model_features': feature_config['mood_model_features']
}

with open('artifacts/dashboard/feature_schema.json', 'w') as f:
    json.dump(dashboard_feature_schema, f, indent=2)

with open('artifacts/dashboard/mood_definition.json', 'w') as f:
    json.dump(mood_definition, f, indent=2)

visual_theme = {
    'moods': {
        'Euphoric': '#F4C95D',
        'Peaceful': '#67C6C3',
        'Aggressive': '#E85D5D',
        'Melancholic': '#6C70B5'
    },
    'cluster_palette': 'Set2'
}

with open('artifacts/dashboard/visual_theme.json', 'w') as f:
    json.dump(visual_theme, f, indent=2)

print('Shared contracts saved.')


Shared contracts saved.


## 8. Build one model manifest

The backend can read this single file to see which model files exist, what features each one expects and what output it produces.

In [9]:
metadata_files = {
    'popularity': 'models/metadata/popularity_regression.json',
    'mood': 'models/metadata/mood_classifier.json',
    'genre': 'models/metadata/genre_classifier.json',
    'cluster': 'models/metadata/emotion_kmeans.json',
    'pca': 'models/metadata/emotion_pca.json',
    'similarity': 'models/metadata/similar_songs_nn.json'
}

metadata = {}

for name, path in metadata_files.items():
    with open(path) as f:
        metadata[name] = json.load(f)

model_manifest = {
    'version': '1.0.0',
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'models': {
        'popularity': {
            'path': 'models/popularity_regression.joblib',
            'input_features': metadata['popularity']['features'],
            'output': 'predicted_popularity',
            'metrics': metadata['popularity']['metrics']
        },
        'mood': {
            'path': 'models/mood_classifier.joblib',
            'input_features': metadata['mood']['features'],
            'output': 'mood_label',
            'metrics': metadata['mood']['metrics']
        },
        'genre': {
            'path': 'models/genre_classifier.joblib',
            'input_features': metadata['genre']['features'],
            'output': 'genre_class_and_probabilities',
            'metrics': metadata['genre']['metrics']
        },
        'cluster': {
            'path': 'models/emotion_kmeans.joblib',
            'input_features': metadata['cluster']['features'],
            'output': 'cluster_id',
            'metrics': metadata['cluster']['metrics']
        },
        'pca': {
            'path': 'models/emotion_pca.joblib',
            'input_features': metadata['pca']['features'],
            'output': 'pca_1_and_pca_2',
            'explained_variance_total': metadata['pca']['explained_variance_total']
        },
        'similarity': {
            'path': 'models/similar_songs_nn.joblib',
            'input_features': metadata['similarity']['features'],
            'output': 'nearest_track_indices',
            'catalog_preferred': 'artifacts/dashboard/similarity_catalog.parquet',
            'catalog_fallback': 'artifacts/dashboard/similarity_catalog.csv.gz'
        }
    },
    'dashboard_tables': {
        'yearly_mood_trends': {
            'preferred_path': 'artifacts/dashboard/yearly_mood_trends.parquet',
            'fallback_path': 'artifacts/dashboard/yearly_mood_trends.csv.gz',
            'grain': 'year'
        },
        'genre_profiles': {
            'preferred_path': 'artifacts/dashboard/genre_profiles.parquet',
            'fallback_path': 'artifacts/dashboard/genre_profiles.csv.gz',
            'grain': 'track_genre'
        },
        'country_mood_trends': {
            'preferred_path': 'artifacts/dashboard/country_mood_trends.parquet',
            'fallback_path': 'artifacts/dashboard/country_mood_trends.csv.gz',
            'grain': 'country + snapshot_date'
        },
        'cluster_points': {
            'preferred_path': 'artifacts/dashboard/cluster_points.parquet',
            'fallback_path': 'artifacts/dashboard/cluster_points.csv.gz',
            'grain': 'track'
        },
        'track_catalog': {
            'preferred_path': 'artifacts/dashboard/track_catalog.parquet',
            'fallback_path': 'artifacts/dashboard/track_catalog.csv.gz',
            'grain': 'track'
        }
    }
}

with open('artifacts/dashboard/model_manifest.json', 'w') as f:
    json.dump(model_manifest, f, indent=2)

model_manifest

{'version': '1.0.0',
 'created_at': '2026-09-23T23:51:28',
 'models': {'popularity': {'path': 'models/popularity_regression.joblib',
   'input_features': ['danceability',
    'energy',
    'key',
    'loudness',
    'mode',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo',
    'duration_ms',
    'time_signature'],
   'output': 'predicted_popularity',
   'metrics': {'mae': 9.085448047200211,
    'mse': 211.0004313561596,
    'rmse': 14.525853894217702,
    'r2': -0.014137182604360365}},
  'mood': {'path': 'models/mood_classifier.joblib',
   'input_features': ['danceability',
    'key',
    'loudness',
    'mode',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'tempo',
    'duration_ms',
    'time_signature'],
   'output': 'mood_label',
   'metrics': {'accuracy': 0.6521254665998105,
    'macro_precision': 0.6194731592334108,
    'macro_recall': 0.5979868230716957,
    'macro_f1': 0.60576727948060

## 9. Final artifact smoke tests

These checks prove that the handoff files are not just present: the saved models are loaded again and used on fresh rows.

In [10]:
validation_checks = {}

expected_model_files = [
    'models/popularity_regression.joblib',
    'models/mood_classifier.joblib',
    'models/genre_classifier.joblib',
    'models/emotion_kmeans.joblib',
    'models/emotion_pca.joblib',
    'models/similar_songs_nn.joblib'
]

validation_checks['all_model_files_exist'] = all(
    os.path.exists(path) for path in expected_model_files
)

# Load every model artifact.
popularity_model = joblib.load('models/popularity_regression.joblib')
mood_model = joblib.load('models/mood_classifier.joblib')
genre_model = joblib.load('models/genre_classifier.joblib')
kmeans_bundle = joblib.load('models/emotion_kmeans.joblib')
pca_bundle = joblib.load('models/emotion_pca.joblib')
similarity_bundle = joblib.load('models/similar_songs_nn.joblib')

validation_checks['all_models_load'] = True

# Use one real catalog row for cluster/PCA/similarity smoke checks.
cluster_features = kmeans_bundle['features']
cluster_input = track_catalog.dropna(subset=cluster_features).iloc[[0]][cluster_features]
cluster_scaled = kmeans_bundle['scaler'].transform(cluster_input)

cluster_prediction = kmeans_bundle['model'].predict(cluster_scaled)[0]
pca_prediction = pca_bundle['model'].transform(
    pca_bundle['scaler'].transform(cluster_input)
)[0]
neighbor_indices = similarity_bundle['model'].kneighbors(
    similarity_bundle['scaler'].transform(cluster_input),
    n_neighbors=5,
    return_distance=False
)[0]

validation_checks['cluster_prediction_valid'] = int(cluster_prediction) >= 0
validation_checks['pca_prediction_has_two_values'] = len(pca_prediction) == 2
validation_checks['similarity_returns_five_rows'] = len(neighbor_indices) == 5
validation_checks['similarity_indices_fit_catalog'] = int(neighbor_indices.max()) < len(similarity_catalog)

# Supervised model smoke checks use the feature orders stored in the manifest.
supervised_source = genre.dropna().iloc[[0]]

pop_features = model_manifest['models']['popularity']['input_features']
mood_features = model_manifest['models']['mood']['input_features']
genre_features = model_manifest['models']['genre']['input_features']

popularity_prediction = popularity_model.predict(supervised_source[pop_features])[0]
mood_prediction = mood_model.predict(supervised_source[mood_features])[0]
genre_prediction = genre_model.predict(supervised_source[genre_features])[0]

validation_checks['popularity_prediction_is_numeric'] = bool(np.isfinite(popularity_prediction))
validation_checks['mood_prediction_valid'] = mood_prediction in ['Euphoric', 'Peaceful', 'Aggressive', 'Melancholic']
validation_checks['genre_prediction_not_empty'] = bool(str(genre_prediction).strip())

# Check that the analytical exports contain no NaN/Inf in required numeric fields.
required_yearly = [
    'mean_valence', 'mean_energy', 'mean_danceability', 'mean_popularity'
]

validation_checks['yearly_export_has_no_missing_required_values'] = not yearly_mood_trends[required_yearly].isna().any().any()
validation_checks['cluster_profiles_exist'] = len(cluster_profiles) > 0
validation_checks['model_manifest_exists'] = os.path.exists('artifacts/dashboard/model_manifest.json')
validation_checks['feature_schema_exists'] = os.path.exists('artifacts/dashboard/feature_schema.json')
validation_checks['visual_theme_exists'] = os.path.exists('artifacts/dashboard/visual_theme.json')
validation_checks['parquet_exports_created'] = parquet_created

validation_status = 'PASS' if all(validation_checks.values()) else 'PASS_WITH_CSV_FALLBACK'

handoff_validation = {
    'status': validation_status,
    'validated_at': datetime.now().isoformat(timespec='seconds'),
    'checks': validation_checks
}

with open('artifacts/dashboard/handoff_validation.json', 'w') as f:
    json.dump(handoff_validation, f, indent=2)

handoff_validation

{'status': 'PASS',
 'validated_at': '2026-09-23T23:51:37',
 'checks': {'all_model_files_exist': True,
  'all_models_load': True,
  'cluster_prediction_valid': True,
  'pca_prediction_has_two_values': True,
  'similarity_returns_five_rows': True,
  'similarity_indices_fit_catalog': True,
  'popularity_prediction_is_numeric': True,
  'mood_prediction_valid': True,
  'genre_prediction_not_empty': True,
  'yearly_export_has_no_missing_required_values': True,
  'cluster_profiles_exist': True,
  'model_manifest_exists': True,
  'feature_schema_exists': True,
  'visual_theme_exists': True,
  'parquet_exports_created': True}}

## 10. Final contents

In [11]:
for filename in sorted(os.listdir('artifacts/dashboard')):
    print(filename)

.gitignore
cluster_points.csv.gz
cluster_points.parquet
cluster_profiles.json
country_mood_trends.csv.gz
country_mood_trends.parquet
feature_schema.json
genre_profiles.csv.gz
genre_profiles.parquet
handoff_validation.json
model_manifest.json
mood_definition.json
similarity_catalog.csv.gz
similarity_catalog.parquet
track_catalog.csv.gz
track_catalog.parquet
visual_theme.json
yearly_mood_trends.csv.gz
yearly_mood_trends.parquet
